In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mlflow
from mlflow.tracking import MlflowClient
import pickle
from glob import glob

import sys
import os
sys.path.append(os.path.abspath("../../"))
from neuralforecast.losses.numpy import mae, mse, rmse, mape, smape, mase

In [ ]:
# Set MLflow experiment
mlflow.set_experiment("/Users/silas.schroeder@sap.com/qtsf")
client = MlflowClient()

# Define experiment label (should match what was used in model-training.ipynb)
EXPERIMENT = "dev"

In [ ]:
# Load test data
df = pd.read_csv("../data/train_prepared.csv", parse_dates=["Date"], low_memory=False)
df = df.sort_values(["Store", "Date"])

delta_days = (max(df["Date"]) - min(df["Date"])).days
train_days = int(delta_days * 0.8)
split_date = min(df["Date"]) + np.timedelta64(train_days,"D")

train_df = df[df["Date"] < split_date].copy()
test_df  = df[df["Date"] >= split_date].copy()

HORIZON = (max(df["Date"]) - np.datetime64(split_date)).days
y_train_global = train_df["Sales"].values
SEASONALITY = 7

print(f"Test period: {test_df['Date'].min().date()} to {test_df['Date'].max().date()}")
print(f"Horizon: {HORIZON} days")

In [ ]:
# Retrieve all runs from MLflow for this experiment
experiment = client.get_experiment_by_name("/Users/silas.schroeder@sap.com/qtsf")
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string=f"tags.experiment = '{EXPERIMENT}'",
    order_by=["start_time DESC"]
)

print(f"Found {len(runs)} runs for experiment '{EXPERIMENT}'")

# Group runs by run_label and model
run_data = {}
for run in runs:
    run_label = run.data.params.get("run_label")
    model = run.data.params.get("model")
    
    if run_label and model:
        if run_label not in run_data:
            run_data[run_label] = {}
        
        run_data[run_label][model] = {
            "metrics": run.data.metrics,
            "params": run.data.params,
            "run_id": run.info.run_id
        }

print(f"\nRun labels found: {list(run_data.keys())}")
for run_label, models in run_data.items():
    print(f"  {run_label}: {list(models.keys())}")

In [ ]:
# Reconstruct all_avg_results and all_seed_results from MLflow
all_avg_results = {}
all_seed_results = {}
all_param_counts = {}
all_training_times = {}

for run_label, models in run_data.items():
    all_avg_results[run_label] = {}
    all_seed_results[run_label] = {}
    all_training_times[run_label] = []
    all_param_counts[run_label] = {}
    
    for model, data in models.items():
        metrics = data["metrics"]
        
        # Extract avg metrics (these are the seed-ensemble metrics)
        all_avg_results[run_label][model] = {
            "MAE": metrics.get("avg_MAE", 0),
            "MSE": metrics.get("avg_MSE", 0),
            "RMSE": metrics.get("avg_RMSE", 0),
            "MAPE": metrics.get("avg_MAPE", 0),
            "MASE": metrics.get("avg_MASE", 0),
        }
        
        # Extract seed statistics
        all_seed_results[run_label][model] = {}
        for metric_name in ["MAE", "MSE", "RMSE", "MAPE", "MASE"]:
            mean_val = metrics.get(f"mean_{metric_name}", 0)
            std_val = metrics.get(f"std_{metric_name}", 0)
            # Reconstruct approximate per-seed values (for visualization)
            # Note: This is an approximation. For exact values, we'd need to log them separately
            all_seed_results[run_label][model][metric_name] = [mean_val] * 3  # assuming 3 seeds
        
        # Extract parameter counts
        param_count = metrics.get("num_parameters")
        if param_count is not None:
            all_param_counts[run_label][model] = int(param_count)
        
        # Extract training times
        mean_time = metrics.get("mean_training_time_s")
        if mean_time is not None and mean_time not in all_training_times[run_label]:
            all_training_times[run_label].append(mean_time)

print("\nReconstructed data structures from MLflow:")
print(f"Run labels: {list(all_avg_results.keys())}")
print(f"Example metrics for first model in first run:")
first_run = list(all_avg_results.keys())[0]
first_model = list(all_avg_results[first_run].keys())[0]
print(f"  {first_run}/{first_model}: {all_avg_results[first_run][first_model]}")

In [ ]:
forecast_dir = 'forecasts'
all_avg_forecasts = {}
all_seed_forecasts = {}

if os.path.exists(forecast_dir):
    # Finde alle forecasts.pkl in Unterordnern (classical, naive, ideal, noisy_sim, etc.)
    forecast_files = glob(os.path.join(forecast_dir, '*', 'forecasts.pkl'))
    
    if forecast_files:
        print(f"✓ Found {len(forecast_files)} forecast files:")
        
        for fpath in forecast_files:
            # Extrahiere run_label aus dem Ordnernamen
            run_label = os.path.basename(os.path.dirname(fpath))
            
            with open(fpath, 'rb') as f:
                forecast_data = pickle.load(f)
            
            # forecast_data kann verschiedene Strukturen haben:
            # 1. Dict mit 'all_avg_forecasts' und 'all_seed_forecasts' keys
            # 2. Direkt ein DataFrame mit Forecasts
            if isinstance(forecast_data, dict):
                if 'all_avg_forecasts' in forecast_data:
                    # Nested structure - extrahiere für dieses run_label
                    all_avg_forecasts[run_label] = forecast_data['all_avg_forecasts'].get(run_label, forecast_data['all_avg_forecasts'])
                    if 'all_seed_forecasts' in forecast_data:
                        all_seed_forecasts[run_label] = forecast_data['all_seed_forecasts'].get(run_label, forecast_data['all_seed_forecasts'])
                else:
                    # Dict ist direkt das forecast DataFrame oder enthält model columns
                    all_avg_forecasts[run_label] = forecast_data
            else:
                # DataFrame direkt
                all_avg_forecasts[run_label] = forecast_data
            
            print(f"  - {run_label}: {fpath}")
            if isinstance(all_avg_forecasts[run_label], pd.DataFrame):
                print(f"    Columns: {list(all_avg_forecasts[run_label].columns)}")
        
        print(f"\n✓ Loaded forecasts for run labels: {list(all_avg_forecasts.keys())}")
        print("  Store-level plots are now available!")
    else:
        print(f"⚠️  Warning: No forecasts.pkl files found in {forecast_dir}/*/")
        print("   Run model-training.ipynb first to generate forecasts.")
else:
    print(f"⚠️  Warning: {forecast_dir}/ directory not found.")
    print("   Run model-training.ipynb first to generate forecasts.")
    print("   Store-level plotting cells will be skipped.")

In [ ]:
# ── Summary table ────────────────────────────────────────────────────────────
print("\nTest-set metrics  (seed-ensemble ★ | seeds mean ± std)")
print("─" * 70)
for run_label in all_avg_results:
    print(f"\n  [{run_label}]")
    for model in sorted(all_avg_results[run_label], key=lambda m: all_avg_results[run_label][m]["MAE"]):
        a = all_avg_results[run_label][model]
        s = all_seed_results[run_label][model]
        print(f"    {model}:")
        for metric in ["MAE", "MSE", "RMSE", "MAPE", "MASE"]:
            print(f"      {metric}: ★{a[metric]:.4f}  |  "
                  f"seeds {np.mean(s[metric]):.4f} ± {np.std(s[metric]):.4f}")

In [ ]:
# ── Box plots: one row per run_label, one subplot per metric ─────────────────
metrics_to_plot = ["MAE", "MSE", "RMSE", "MAPE", "MASE"]
n_runs = len(all_seed_results)

fig, axes = plt.subplots(n_runs, len(metrics_to_plot),
                         figsize=(4 * len(metrics_to_plot), 4 * n_runs),
                         squeeze=False)
fig.suptitle(
    "Per-seed metrics (box/dots) vs. seed-averaged forecast (★)\n"
    "Rows = run (classical / quantum device),  Columns = metric",
    fontsize=11, y=1.01
)

for row, (run_label, seed_results) in enumerate(all_seed_results.items()):
    model_names = list(seed_results.keys())
    for col, metric in enumerate(metrics_to_plot):
        ax = axes[row][col]
        data = [seed_results[m][metric] for m in model_names]
        ax.boxplot(data, tick_labels=model_names, patch_artist=True, notch=False,
                   medianprops=dict(color="black", linewidth=2))
        for i, values in enumerate(data, start=1):
            ax.scatter([i] * len(values), values, color="black", zorder=5, s=20,
                       label="individual seed" if (i == 1 and col == 0) else "")
        for i, m in enumerate(model_names, start=1):
            ax.scatter(i, all_avg_results[run_label][m][metric],
                       marker="*", color="red", zorder=6, s=150,
                       label="seed-avg ★" if (i == 1 and col == 0) else "")
        if col == 0:
            ax.set_ylabel(f"{run_label}\n{metric}", fontsize=9)
        else:
            ax.set_ylabel(metric, fontsize=9)
        if row == 0:
            ax.set_title(metric)
        ax.grid(True, axis="y", alpha=0.3)
        if col == 0:
            ax.legend(loc="upper right", fontsize=7)

plt.tight_layout()
plt.show()

In [ ]:
# ── MAE comparison bar chart across all runs ─────────────────────────────────
fig, ax = plt.subplots(figsize=(max(8, 2 * len(all_avg_results)), 4))
x = np.arange(len(all_avg_results))
run_labels = list(all_avg_results.keys())
all_model_names = sorted({m for r in all_avg_results.values() for m in r})
bar_width = 0.8 / len(all_model_names)

for j, model in enumerate(all_model_names):
    maes = [all_avg_results[rl].get(model, {}).get("MAE", float("nan"))
            for rl in run_labels]
    ax.bar(x + j * bar_width, maes, width=bar_width, label=model, alpha=0.85)

ax.set_xticks(x + bar_width * (len(all_model_names) - 1) / 2)
ax.set_xticklabels(run_labels)
ax.set_ylabel("MAE (seed-ensemble)")
ax.set_title("MAE comparison: classical baseline vs. quantum devices")
ax.legend()
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Check if forecast data is available for store-level plots
if not all_avg_forecasts:
    print("⚠️  Skipping store-level plots: forecast data not available.")
    print("   Run model-training.ipynb and ensure forecasts.pkl is created.")
else:
    # ── Helper: compute per-store MAE for a specific model pair ──
    def store_mae_for_model_pair(classical_model, quantum_model):
        """Returns a Series indexed by Store with mean MAE for the naive + classical + quantum model."""
        per_store = []

        # Naive
        naive_fc = all_avg_forecasts["naive"]
        comp = test_df[["Store", "Date", "Sales"]].merge(naive_fc, on=["Store", "Date"], how="inner")
        store_err = comp.groupby("Store").apply(
            lambda g: (g["Sales"] - g["Naive"]).abs().mean(), include_groups=False
        )
        per_store.append(store_err)

        # Classical model
        if "classical" in all_avg_forecasts:
            classical_fc = all_avg_forecasts["classical"]
            if classical_model in classical_fc.columns:
                comp = test_df[["Store", "Date", "Sales"]].merge(classical_fc, on=["Store", "Date"], how="inner")
                store_err = comp.groupby("Store").apply(
                    lambda g: (g["Sales"] - g[classical_model]).abs().mean(), include_groups=False
                )
                per_store.append(store_err)

        # Quantum model (take first quantum device found)
        quantum_labels = [rl for rl in all_avg_forecasts if rl not in ("naive", "classical")]
        if quantum_labels and quantum_model:
            quantum_fc = all_avg_forecasts[quantum_labels[0]]
            if quantum_model in quantum_fc.columns:
                comp = test_df[["Store", "Date", "Sales"]].merge(quantum_fc, on=["Store", "Date"], how="inner")
                store_err = comp.groupby("Store").apply(
                    lambda g: (g["Sales"] - g[quantum_model]).abs().mean(), include_groups=False
                )
                per_store.append(store_err)

        return pd.concat(per_store, axis=1).mean(axis=1)


    # ── Helper: plot one store with actual + naive + classical model + quantum model ──
    def plot_store_model_pair(ax, store_id, classical_model, quantum_model, title):
        store_actual = test_df[test_df["Store"] == store_id].sort_values("Date")

        # Plot actual
        ax.plot(store_actual["Date"], store_actual["Sales"],
                label="Actual", color="black", linewidth=2)

        # Plot naive
        naive_fc = all_avg_forecasts["naive"]
        store_naive = naive_fc[naive_fc["Store"] == store_id].sort_values("Date")
        if not store_naive.empty:
            naive_mae = (store_actual.set_index("Date")["Sales"]
                        .reindex(store_naive["Date"].values) - store_naive["Naive"].values
                        ).abs().mean()
            ax.plot(store_naive["Date"], store_naive["Naive"],
                    label=f"Naive (MAE {naive_mae:.0f})",
                    color="gray", linestyle="--", linewidth=1.5)

        # Plot classical model
        if "classical" in all_avg_forecasts:
            classical_fc = all_avg_forecasts["classical"]
            if classical_model in classical_fc.columns:
                store_classical = classical_fc[classical_fc["Store"] == store_id].sort_values("Date")
                if not store_classical.empty:
                    classical_mae = (store_actual.set_index("Date")["Sales"]
                                   .reindex(store_classical["Date"].values) - store_classical[classical_model].values
                                   ).abs().mean()
                    ax.plot(store_classical["Date"], store_classical[classical_model],
                            label=f"{classical_model} (MAE {classical_mae:.0f})",
                            color="blue", linestyle="-", linewidth=1.5)

        # Plot quantum model (take first quantum device found)
        quantum_labels = [rl for rl in all_avg_forecasts if rl not in ("naive", "classical")]
        if quantum_labels and quantum_model:
            quantum_fc = all_avg_forecasts[quantum_labels[0]]
            if quantum_model in quantum_fc.columns:
                store_quantum = quantum_fc[quantum_fc["Store"] == store_id].sort_values("Date")
                if not store_quantum.empty:
                    quantum_mae = (store_actual.set_index("Date")["Sales"]
                                 .reindex(store_quantum["Date"].values) - store_quantum[quantum_model].values
                                 ).abs().mean()
                    ax.plot(store_quantum["Date"], store_quantum[quantum_model],
                            label=f"{quantum_model} (MAE {quantum_mae:.0f})",
                            color="red", linestyle="-", linewidth=1.5)

        ax.set_title(title)
        ax.legend(fontsize=9, loc="upper left")
        ax.grid(True, alpha=0.3)
        ax.set_ylabel("Sales")


    # ── Define model pairs ────────────────────────────────────────────────────────
    model_pairs = [
        ("DLinear", "QDLinear"),
        ("NHITS", "QNHITS"),
        ("PatchTST", "QPatchTST"),
        ("TimesNet", "QTimesNet"),
    ]

    # ── Find best/worst stores for each model pair ───────────────────────────────
    print("Finding best/worst stores for each model pair:")
    pair_stores = []
    for classical_model, quantum_model in model_pairs:
        store_mae = store_mae_for_model_pair(classical_model, quantum_model)
        best_store = store_mae.idxmin()
        worst_store = store_mae.idxmax()
        pair_stores.append((best_store, worst_store))
        print(f"  {classical_model}/{quantum_model} — "
              f"best: {best_store} (MAE {store_mae[best_store]:.1f}), "
              f"worst: {worst_store} (MAE {store_mae[worst_store]:.1f})")

    # ── Build figure: 4 model pairs × 2 (best/worst) = 4 rows × 2 cols ───────────
    fig, axes = plt.subplots(4, 2, figsize=(20, 24), squeeze=False)
    fig.suptitle("Model Pair Comparisons: Best & Worst Store Forecasts\n"
                 "Each plot shows: Actual + Naive + Classical Model + Quantum Model",
                 fontsize=14)

    for idx, ((classical_model, quantum_model), (best_store, worst_store)) in enumerate(zip(model_pairs, pair_stores)):
        # Best store
        plot_store_model_pair(axes[idx][0], best_store, classical_model, quantum_model,
                             f"{classical_model} / {quantum_model} — Best Store {best_store}")

        # Worst store
        plot_store_model_pair(axes[idx][1], worst_store, classical_model, quantum_model,
                             f"{classical_model} / {quantum_model} — Worst Store {worst_store}")

    plt.tight_layout()
    plt.show()